In [43]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
model=ChatOpenAI()


In [44]:
class review(TypedDict):
    review_txt:str
    sentiment:Literal['positive','negative']
    response:str
    issue:str
    solution:str

class sentiment(BaseModel):
    sentiment:Literal['positive','negative']

class diagnosis(BaseModel):
    issue:str
    solution:str


sentiment_model=model.with_structured_output(sentiment)
diagnosis_model=model.with_structured_output(diagnosis)



d:\AgenticAI_Learning\LangGraph\venv\Lib\site-packages\langchain_openai\chat_models\base.py:2418: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [45]:
def findsenti(state:review)->review:
    prompt=f"Analyze the sentiment of the following review and classify it as positive, negative:\n{state['review_txt']}"
    response=sentiment_model.invoke(prompt)
    state['sentiment']=response.sentiment
    return {'sentiment':state['sentiment']}

In [46]:
def rundiag(state:review)->review:
    prompt=f"Diagnose the following negative review and provide a solution:\n{state['review_txt']}"
    response=diagnosis_model.invoke(prompt)
    state['issue']=response.issue
    state['solution']=response.solution
    return {'issue':state['issue'],'solution':state['solution']}
def posresponse(state:review)->review:
    prompt=f"Generate a positive response to the following review in less tokens:\n{state['review_txt']}"
    response=model.invoke(prompt)
    state['response']=response.content
    return {'response':state['response']}

def negresponse(state:review)->review:
    prompt=f"Generate a response to the following negative review addressing the issue and providing a solution in less tokens:\n{state['review_txt']}\nIssue: {state['issue']}\nSolution: {state['solution']}"
    response=model.invoke(prompt)
    state['response']=response.content
    return {'response':state['response']}

def check(state:review)->Literal['Positive_response','Run_diagnosis']:
    if state['sentiment']=='positive':
        return 'Positive_response'
    else:
        return 'Run_diagnosis'
    

In [47]:
graph=StateGraph(review)
graph.add_node('Find_sentiment',findsenti)
graph.add_node('Run_diagnosis',rundiag)
graph.add_node('Positive_response',posresponse)
graph.add_node('Negative_response',negresponse)


graph.add_edge(START,'Find_sentiment')
graph.add_conditional_edges('Find_sentiment',check)
graph.add_edge('Run_diagnosis','Negative_response')
graph.add_edge('Positive_response',END)
graph.add_edge('Negative_response',END) 
workflow=graph.compile()

initial_state={'review_txt':"The product quality is terrible and the customer service is unhelpful. I am extremely disappointed with my purchase and will not be buying from this company again."}
workflow.invoke(initial_state)


{'review_txt': 'The product quality is terrible and the customer service is unhelpful. I am extremely disappointed with my purchase and will not be buying from this company again.',
 'sentiment': 'negative',
 'response': 'We apologize for your negative experience. Please contact our customer service to address the product quality and service issues. We will work to resolve the problem and ensure your satisfaction. Thank you for your feedback.',
 'issue': 'product quality is terrible and customer service is unhelpful',
 'solution': "contact the company's customer service to address the quality issues and customer service concerns. Consider asking for a refund or exchange for a better product. If the company is unresponsive, consider leaving a detailed review online to share your experience and inform other potential customers."}